# 02 · Maestro financiero e integración contable

## Objetivo

Transformar el plan general contable en un maestro financiero sencillo y unirlo a la contabilidad limpia.

Este notebook:

- clasifica las cuentas en Balance, PyG o Patrimonio Neto;
- asigna masas y categorías financieras útiles;
- identifica clientes, proveedores, tesorería, deuda, intereses y amortizaciones;
- conserva el centro de coste;
- marca las cuentas que requieren revisión;
- exporta tablas preparadas para construir estados financieros y ratios.

> Las reglas se basan en los grupos y subgrupos del PGC y pueden revisarse desde un único bloque.

In [1]:
# ==========================================================
# 1. LIBRERÍAS Y RUTAS
# ==========================================================

from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


# ==========================================================
# RUTAS DEL PROYECTO
# ==========================================================

CWD = Path.cwd()

if (CWD / "data").exists():
    BASE_DIR = CWD
elif (CWD.parent / "data").exists():
    BASE_DIR = CWD.parent
else:
    raise FileNotFoundError(
        "No se ha podido localizar la raíz del proyecto. "
        "Debe existir una carpeta 'data/' en la raíz."
    )

PROCESSED_DIR = BASE_DIR / "data" / "processed"
INPUT_DIR = PROCESSED_DIR / "01"
OUTPUT_DIR = PROCESSED_DIR / "02"

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


RUTAS = {
    "contabilidad": (
        INPUT_DIR
        / "01_contabilidad_limpia.csv"
    ),
    "plan_n1": (
        INPUT_DIR
        / "01_plan_cuentas_n1.csv"
    ),
    "plan_n2": (
        INPUT_DIR
        / "01_plan_cuentas_n2.csv"
    ),
    "plan_n5": (
        INPUT_DIR
        / "01_plan_cuentas_n5.csv"
    ),
}


# ==========================================================
# COMPROBACIÓN DE ARCHIVOS
# ==========================================================

for nombre, ruta in RUTAS.items():

    if not ruta.exists():
        raise FileNotFoundError(
            f"No se encontró '{nombre}':\n{ruta}\n"
            "Ejecuta primero el NB1."
        )


print(f"Raíz del proyecto: {BASE_DIR}")
print(f"Entrada NB2:        {INPUT_DIR}")
print(f"Salida NB2:         {OUTPUT_DIR}")
print("Archivos de entrada encontrados correctamente.")

Raíz del proyecto: C:\Users\moned\Downloads\Data_Science_Evolve\analisis_ratios
Entrada NB2:        C:\Users\moned\Downloads\Data_Science_Evolve\analisis_ratios\data\processed\01
Salida NB2:         C:\Users\moned\Downloads\Data_Science_Evolve\analisis_ratios\data\processed\02
Archivos de entrada encontrados correctamente.


In [2]:
# ==========================================================
# 2. CARGA Y VALIDACIÓN DE ARCHIVOS
# ==========================================================

# ==========================================================
# COLUMNAS NECESARIAS
# ==========================================================

COLUMNAS_CONTABILIDAD = [
    "periodo",
    "cuenta_n5",
    "centro_coste",
    "importe",
]

COLUMNAS_PLAN_N1 = [
    "cuenta_n1",
    "descripcion_n1",
]

COLUMNAS_PLAN_N2 = [
    "cuenta_n2",
    "descripcion_n2",
]

COLUMNAS_PLAN_N5 = [
    "cuenta_n5",
    "descripcion_n5",
]


# ==========================================================
# FUNCIÓN DE VALIDACIÓN
# ==========================================================

def validar_columnas_csv(
    ruta,
    columnas_esperadas
):

    disponibles = pd.read_csv(
        ruta,
        nrows=0
    ).columns.tolist()

    faltantes = [
        columna
        for columna in columnas_esperadas
        if columna not in disponibles
    ]

    if faltantes:
        raise ValueError(
            f"El archivo '{ruta.name}' no contiene "
            f"las columnas necesarias: {faltantes}"
        )


# ==========================================================
# VALIDACIÓN PREVIA
# ==========================================================

validar_columnas_csv(
    RUTAS["contabilidad"],
    COLUMNAS_CONTABILIDAD
)

validar_columnas_csv(
    RUTAS["plan_n1"],
    COLUMNAS_PLAN_N1
)

validar_columnas_csv(
    RUTAS["plan_n2"],
    COLUMNAS_PLAN_N2
)

validar_columnas_csv(
    RUTAS["plan_n5"],
    COLUMNAS_PLAN_N5
)


# ==========================================================
# CARGA
# ==========================================================

contabilidad = pd.read_csv(
    RUTAS["contabilidad"],
    dtype={
        "cuenta_n0": "string",
        "cuenta_n1": "string",
        "cuenta_n2": "string",
        "cuenta_n5": "string",
        "centro_coste": "string",
        "intercompany": "string",
    },
    parse_dates=["periodo"]
)

plan_n1 = pd.read_csv(
    RUTAS["plan_n1"],
    dtype="string"
)

plan_n2 = pd.read_csv(
    RUTAS["plan_n2"],
    dtype="string"
)

plan_n5 = pd.read_csv(
    RUTAS["plan_n5"],
    dtype="string"
)


# ==========================================================
# CONTROL
# ==========================================================

print(f"Movimientos contables: {len(contabilidad):,}")
print(f"Plan N1:               {len(plan_n1):,} cuentas")
print(f"Plan N2:               {len(plan_n2):,} cuentas")
print(f"Plan N5:               {len(plan_n5):,} cuentas")

print(
    f"Periodo: "
    f"{contabilidad['periodo'].min():%Y-%m} → "
    f"{contabilidad['periodo'].max():%Y-%m}"
)

print("Carga y estructura de entrada validadas correctamente.")

Movimientos contables: 78,688
Plan N1:               9 cuentas
Plan N2:               79 cuentas
Plan N5:               1,565 cuentas
Periodo: 2021-01 → 2023-12
Carga y estructura de entrada validadas correctamente.


In [3]:
# ==========================================================
# 3. REGLAS DEL MAESTRO FINANCIERO
# ==========================================================

def clasificar_cuenta(cuenta):

    c1 = cuenta[:1]
    c2 = cuenta[:2]
    c3 = cuenta[:3]

    estado = "CLASIFICADA"
    tipo_estado = "Sin clasificar"
    masa = "Sin clasificar"
    categoria = "Sin clasificar"
    subcategoria = "Otras partidas"
    naturaleza = "N/A"
    circulante = "N/A"
    driver = "Otros"


    # ======================================================
    # BALANCE Y PATRIMONIO NETO
    # ======================================================

    if c2 in {"10", "11", "12", "13"}:

        tipo_estado = "Balance"
        masa = "Patrimonio neto"
        categoria = "Patrimonio neto"

        subcategoria = (
            "Capital y reservas"
            if c2 in {"10", "11"}
            else "Resultados y ajustes"
        )

        naturaleza = "Acreedora"
        circulante = "No corriente"
        driver = "Patrimonio"


    elif c2 == "14":

        tipo_estado = "Balance"
        masa = "Pasivo no corriente"
        categoria = "Provisiones"
        subcategoria = "Provisiones a largo plazo"
        naturaleza = "Acreedora"
        circulante = "No corriente"
        driver = "Deuda y riesgo"


    elif c2 in {"15", "16", "17", "18"}:

        tipo_estado = "Balance"
        masa = "Pasivo no corriente"

        categoria = (
            "Deuda financiera"
            if (
                c2 in {"15", "17"}
                or c3 in {"160", "162", "163"}
            )
            else "Deudas vinculadas y otros pasivos"
        )

        subcategoria = "Deuda a largo plazo"
        naturaleza = "Acreedora"
        circulante = "No corriente"
        driver = "Deuda"


    elif c2 == "19":

        tipo_estado = "Balance"
        masa = "Patrimonio neto"
        categoria = "Situaciones transitorias de financiación"
        subcategoria = "Capital pendiente"
        naturaleza = "Deudora"
        circulante = "No corriente"
        driver = "Patrimonio"


    elif c2 in {"20", "21", "22", "23"}:

        tipo_estado = "Balance"
        masa = "Activo no corriente"
        categoria = "Inmovilizado"

        subcategoria = {
            "20": "Inmovilizado intangible",
            "21": "Inmovilizado material",
            "22": "Inversiones inmobiliarias",
            "23": "Inmovilizado en curso",
        }[c2]

        naturaleza = "Deudora"
        circulante = "No corriente"
        driver = "Inversión"


    elif c2 in {"24", "25", "26"}:

        tipo_estado = "Balance"
        masa = "Activo no corriente"
        categoria = "Inversiones financieras"
        subcategoria = "Inversiones y depósitos a largo plazo"
        naturaleza = "Deudora"
        circulante = "No corriente"
        driver = "Inversión"


    elif c2 in {"28", "29"}:

        tipo_estado = "Balance"
        masa = "Activo no corriente"
        categoria = "Correcciones de activo"

        subcategoria = (
            "Amortización acumulada"
            if c2 == "28"
            else "Deterioro de activo no corriente"
        )

        naturaleza = "Acreedora"
        circulante = "No corriente"
        driver = "Inversión"


    elif c1 == "3":

        tipo_estado = "Balance"
        masa = "Activo corriente"
        categoria = "Existencias"

        subcategoria = (
            "Deterioro de existencias"
            if c2 == "39"
            else "Existencias"
        )

        naturaleza = (
            "Acreedora"
            if c2 == "39"
            else "Deudora"
        )

        circulante = "Corriente"
        driver = "Existencias"


    elif c2 in {"40", "41"}:

        tipo_estado = "Balance"
        masa = "Pasivo corriente"
        categoria = "Proveedores y acreedores"

        subcategoria = (
            "Proveedores"
            if c2 == "40"
            else "Acreedores varios"
        )

        naturaleza = "Acreedora"
        circulante = "Corriente"
        driver = "Proveedores"


    elif c2 in {"43", "44"}:

        tipo_estado = "Balance"
        masa = "Activo corriente"
        categoria = "Clientes y deudores"

        subcategoria = (
            "Clientes"
            if c2 == "43"
            else "Deudores varios"
        )

        naturaleza = "Deudora"
        circulante = "Corriente"
        driver = "Clientes"


    elif c2 == "45":

        tipo_estado = "Balance"
        masa = "Activo no corriente"
        categoria = "Clientes y deudores"
        subcategoria = "Clientes a largo plazo"
        naturaleza = "Deudora"
        circulante = "No corriente"
        driver = "Clientes"

    elif c2 == "46":

        tipo_estado = "Balance"
        categoria = "Personal"
        circulante = "Corriente"
        driver = "Personal"

        if c3 == "460":

            masa = "Activo corriente"
            subcategoria = "Anticipos de remuneraciones"
            naturaleza = "Deudora"

        elif c3 in {"465", "466"}:

            masa = "Pasivo corriente"
            subcategoria = "Remuneraciones pendientes de pago"
            naturaleza = "Acreedora"

        else:

            masa = "Activo/Pasivo corriente"
            subcategoria = "Otros saldos con personal"
            naturaleza = "Variable"
            estado = "REVISAR"


    elif c2 == "47":

        tipo_estado = "Balance"
        circulante = "Corriente"
        driver = "Impuestos"

        if c3 in {"470", "471", "472", "473", "474"}:

            masa = "Activo corriente"
            categoria = "Administraciones públicas deudoras"
            subcategoria = "Créditos fiscales"
            naturaleza = "Deudora"

        elif c3 in {"475", "476", "477", "479"}:

            masa = "Pasivo corriente"
            categoria = "Administraciones públicas acreedoras"
            subcategoria = "Deudas fiscales y sociales"
            naturaleza = "Acreedora"

        else:

            estado = "REVISAR"
            masa = "Activo/Pasivo corriente"
            categoria = "Administraciones públicas"
            subcategoria = "Administraciones públicas"
            naturaleza = "Variable"


    elif c2 == "48":

        tipo_estado = "Balance"
        circulante = "Corriente"
        driver = "Periodificaciones"

        if c3 == "480":

            masa = "Activo corriente"
            categoria = "Periodificaciones"
            subcategoria = "Gastos anticipados"
            naturaleza = "Deudora"

        elif c3 == "485":

            masa = "Pasivo corriente"
            categoria = "Periodificaciones"
            subcategoria = "Ingresos anticipados"
            naturaleza = "Acreedora"

        else:

            estado = "REVISAR"
            masa = "Activo/Pasivo corriente"
            categoria = "Periodificaciones"
            naturaleza = "Variable"


    elif c2 == "49":

        tipo_estado = "Balance"
        masa = "Activo corriente"
        categoria = "Correcciones de activo"
        subcategoria = "Deterioro de créditos comerciales"
        naturaleza = "Acreedora"
        circulante = "Corriente"
        driver = "Clientes"


    elif c2 in {"50", "51", "52"}:

        tipo_estado = "Balance"
        masa = "Pasivo corriente"
        categoria = "Deuda financiera"
        subcategoria = "Deuda a corto plazo"
        naturaleza = "Acreedora"
        circulante = "Corriente"
        driver = "Deuda"


    elif c2 in {"53", "54"}:

        tipo_estado = "Balance"
        masa = "Activo corriente"
        categoria = "Inversiones financieras"
        subcategoria = "Inversiones financieras a corto plazo"
        naturaleza = "Deudora"
        circulante = "Corriente"
        driver = "Inversión"


    elif c2 == "55":

        tipo_estado = "Balance"
        circulante = "Corriente"

        if c3 == "551":

            masa = "Activo corriente"
            categoria = "Cuentas con vinculadas"
            subcategoria = "Créditos y cuentas corrientes"
            naturaleza = "Deudora"
            driver = "Clientes"

        elif c3 in {"552", "555"}:

            masa = "Pasivo corriente"
            categoria = "Cuentas con vinculadas y partidas pendientes"
            subcategoria = "Deudas y partidas pendientes"
            naturaleza = "Acreedora"
            driver = "Deuda"

        elif c3 == "557":

            masa = "Patrimonio neto"
            categoria = "Patrimonio neto"
            subcategoria = "Dividendo a cuenta"
            naturaleza = "Deudora"
            driver = "Patrimonio"

        else:

            estado = "REVISAR"
            masa = "Activo/Pasivo corriente"
            categoria = "Otras cuentas no bancarias"
            naturaleza = "Variable"


    elif c2 == "56":

        tipo_estado = "Balance"
        circulante = "Corriente"
        driver = "Otros"

        if c3 in {"565", "566"}:

            masa = "Activo corriente"
            categoria = "Fianzas y depósitos"
            subcategoria = "Fianzas constituidas"
            naturaleza = "Deudora"

        elif c3 in {"560", "561"}:

            masa = "Pasivo corriente"
            categoria = "Fianzas y depósitos"
            subcategoria = "Fianzas recibidas"
            naturaleza = "Acreedora"

        else:

            estado = "REVISAR"
            masa = "Activo/Pasivo corriente"
            categoria = "Fianzas y depósitos"
            naturaleza = "Variable"


    elif c2 == "57":

        tipo_estado = "Balance"
        masa = "Activo corriente"
        categoria = "Tesorería"
        subcategoria = "Caja y bancos"
        naturaleza = "Deudora"
        circulante = "Corriente"
        driver = "Caja"


    elif c2 == "58":

        tipo_estado = "Balance"
        circulante = "Corriente"

        if c3 == "580":

            masa = "Activo corriente"
            categoria = "Activos mantenidos para la venta"
            subcategoria = "Activos mantenidos para la venta"
            naturaleza = "Deudora"

        elif c3 == "587":

            masa = "Pasivo corriente"
            categoria = "Deudas vinculadas"
            subcategoria = "Pasivos asociados"
            naturaleza = "Acreedora"

        else:

            estado = "REVISAR"
            masa = "Activo/Pasivo corriente"
            categoria = "Activos y pasivos mantenidos para la venta"
            naturaleza = "Variable"

        driver = "Otros"


    elif c2 == "59":

        tipo_estado = "Balance"
        masa = "Activo corriente"
        categoria = "Correcciones de activo"
        subcategoria = "Deterioro de inversiones financieras"
        naturaleza = "Acreedora"
        circulante = "Corriente"
        driver = "Inversión"


    # ======================================================
    # PÉRDIDAS Y GANANCIAS
    # ======================================================

    elif c1 == "6":

        tipo_estado = "PyG"
        masa = "Gastos"
        naturaleza = "Deudora"
        circulante = "N/A"

        mapa_gastos = {
            "60": (
                "Aprovisionamientos",
                "Compras",
                "Margen bruto"
            ),
            "61": (
                "Aprovisionamientos",
                "Variación de existencias",
                "Margen bruto"
            ),
            "62": (
                "Gastos operativos",
                "Servicios exteriores",
                "EBITDA"
            ),
            "63": (
                "Gastos operativos",
                "Tributos",
                "EBITDA"
            ),
            "64": (
                "Gastos de personal",
                "Gastos de personal",
                "Personal"
            ),
            "65": (
                "Gastos operativos",
                "Otros gastos de gestión",
                "EBITDA"
            ),
            "66": (
                "Resultado financiero",
                "Gastos financieros",
                "Intereses"
            ),
            "67": (
                "Resultados no recurrentes",
                "Pérdidas y gastos excepcionales",
                "No recurrente"
            ),
            "68": (
                "Amortizaciones",
                "Dotaciones para amortizaciones",
                "Amortización"
            ),
            "69": (
                "Deterioros y provisiones",
                "Deterioros y otras dotaciones",
                "Deterioro"
            ),
        }

        categoria, subcategoria, driver = (
            mapa_gastos.get(
                c2,
                (
                    "Otros gastos",
                    "Otros gastos",
                    "Otros"
                )
            )
        )


    elif c1 == "7":

        tipo_estado = "PyG"
        masa = "Ingresos"
        naturaleza = "Acreedora"
        circulante = "N/A"

        mapa_ingresos = {
            "70": (
                "Ingresos de explotación",
                "Ventas y prestación de servicios",
                "Ventas"
            ),
            "71": (
                "Ingresos de explotación",
                "Variación de existencias",
                "Margen bruto"
            ),
            "73": (
                "Ingresos de explotación",
                "Trabajos para la empresa",
                "EBITDA"
            ),
            "74": (
                "Ingresos de explotación",
                "Subvenciones de explotación",
                "EBITDA"
            ),
            "75": (
                "Ingresos de explotación",
                "Otros ingresos de gestión",
                "EBITDA"
            ),
            "76": (
                "Resultado financiero",
                "Ingresos financieros",
                "Intereses"
            ),
            "77": (
                "Resultados no recurrentes",
                "Beneficios e ingresos excepcionales",
                "No recurrente"
            ),
            "79": (
                "Deterioros y provisiones",
                "Reversiones de deterioros y provisiones",
                "Deterioro"
            ),
        }

        categoria, subcategoria, driver = (
            mapa_ingresos.get(
                c2,
                (
                    "Otros ingresos",
                    "Otros ingresos",
                    "Otros"
                )
            )
        )


    elif c1 in {"8", "9"}:

        tipo_estado = "Patrimonio neto"
        masa = "Ingresos y gastos en patrimonio neto"
        categoria = "Ajustes en patrimonio neto"

        subcategoria = (
            "Grupo 8"
            if c1 == "8"
            else "Grupo 9"
        )

        naturaleza = "Variable"
        circulante = "N/A"
        driver = "Patrimonio"


    else:

        estado = "REVISAR"


    return pd.Series({
        "tipo_estado": tipo_estado,
        "masa_financiera": masa,
        "categoria_financiera": categoria,
        "subcategoria_financiera": subcategoria,
        "naturaleza": naturaleza,
        "circulante": circulante,
        "driver_negocio": driver,
        "estado_clasificacion": estado,
    })

In [4]:
# ==========================================================
# 4. CONSTRUCCIÓN DEL MAESTRO FINANCIERO
# ==========================================================

# ==========================================================
# NORMALIZACIÓN DE CLAVES
# ==========================================================

plan_n5["cuenta_n5"] = (
    plan_n5["cuenta_n5"]
    .astype("string")
    .str.strip()
    .str.zfill(6)
)

plan_n1["cuenta_n1"] = (
    plan_n1["cuenta_n1"]
    .astype("string")
    .str.strip()
)

plan_n2["cuenta_n2"] = (
    plan_n2["cuenta_n2"]
    .astype("string")
    .str.strip()
    .str.zfill(2)
)


# ==========================================================
# JERARQUÍA DESDE N5
# ==========================================================

plan_n5["cuenta_n1"] = (
    plan_n5["cuenta_n5"]
    .str[:1]
)

plan_n5["cuenta_n2"] = (
    plan_n5["cuenta_n5"]
    .str[:2]
)


# ==========================================================
# CONSTRUCCIÓN
# ==========================================================

maestro_financiero = (
    plan_n5
    .merge(
        plan_n2,
        on="cuenta_n2",
        how="left",
        validate="many_to_one"
    )
    .merge(
        plan_n1,
        on="cuenta_n1",
        how="left",
        validate="many_to_one"
    )
)

clasificacion = (
    maestro_financiero[
        "cuenta_n5"
    ]
    .apply(
        clasificar_cuenta
    )
)

maestro_financiero = pd.concat(
    [
        maestro_financiero,
        clasificacion,
    ],
    axis=1
)


# ==========================================================
# ORDEN FINAL
# ==========================================================

columnas_maestro = [
    "cuenta_n1",
    "descripcion_n1",
    "cuenta_n2",
    "descripcion_n2",
    "cuenta_n5",
    "descripcion_n5",
    "tipo_estado",
    "masa_financiera",
    "categoria_financiera",
    "subcategoria_financiera",
    "naturaleza",
    "circulante",
    "driver_negocio",
    "estado_clasificacion",
]

maestro_financiero = (
    maestro_financiero[
        columnas_maestro
    ]
    .sort_values(
        "cuenta_n5"
    )
    .reset_index(drop=True)
)


# ==========================================================
# VALIDACIÓN
# ==========================================================

if maestro_financiero[
    "cuenta_n5"
].duplicated().any():

    raise ValueError(
        "El maestro financiero contiene cuentas N5 duplicadas."
    )


print(
    f"Cuentas en maestro financiero: "
    f"{len(maestro_financiero):,}"
)

print(
    maestro_financiero[
        "estado_clasificacion"
    ].value_counts()
)

display(
    maestro_financiero.head()
)

Cuentas en maestro financiero: 1,565
estado_clasificacion
CLASIFICADA    1565
Name: count, dtype: int64


,cuenta_n1,descripcion_n1,cuenta_n2,descripcion_n2,cuenta_n5,descripcion_n5,tipo_estado,masa_financiera,categoria_financiera,subcategoria_financiera,naturaleza,circulante,driver_negocio,estado_clasificacion
0,1,FINANCIACION BASICA,10,CAPITAL,100000,Capital Social,Balance,Patrimonio neto,Patrimonio neto,Capital y reservas,Acreedora,No corriente,Patrimonio,CLASIFICADA
1,1,FINANCIACION BASICA,10,CAPITAL,100001,Capital Social por ampliaciones de Capital,Balance,Patrimonio neto,Patrimonio neto,Capital y reservas,Acreedora,No corriente,Patrimonio,CLASIFICADA
2,1,FINANCIACION BASICA,10,CAPITAL,103000,Socios por desembolsos no exigidos,Balance,Patrimonio neto,Patrimonio neto,Capital y reservas,Acreedora,No corriente,Patrimonio,CLASIFICADA
3,1,FINANCIACION BASICA,11,RESERVAS Y OTROS INSTRUMENTOS DE PATRIMONIO,110000,Prima Emisión,Balance,Patrimonio neto,Patrimonio neto,Capital y reservas,Acreedora,No corriente,Patrimonio,CLASIFICADA
4,1,FINANCIACION BASICA,11,RESERVAS Y OTROS INSTRUMENTOS DE PATRIMONIO,112000,Reserva legal,Balance,Patrimonio neto,Patrimonio neto,Capital y reservas,Acreedora,No corriente,Patrimonio,CLASIFICADA


In [5]:
# ==========================================================
# 5. INTEGRACIÓN CON LA CONTABILIDAD
# ==========================================================

columnas_integracion = [
    "cuenta_n5",
    "descripcion_n5",
    "tipo_estado",
    "masa_financiera",
    "categoria_financiera",
    "subcategoria_financiera",
    "naturaleza",
    "circulante",
    "driver_negocio",
    "estado_clasificacion",
]

contabilidad_financiera = (
    contabilidad
    .merge(
        maestro_financiero[
            columnas_integracion
        ],
        on="cuenta_n5",
        how="left",
        validate="many_to_one"
    )
)


# ==========================================================
# CONTROL DE CUENTAS SIN MAESTRO
# ==========================================================

columnas_texto = [
    "descripcion_n5",
    "tipo_estado",
    "masa_financiera",
    "categoria_financiera",
    "subcategoria_financiera",
    "naturaleza",
    "circulante",
    "driver_negocio",
    "estado_clasificacion",
]

contabilidad_financiera[
    columnas_texto
] = (
    contabilidad_financiera[
        columnas_texto
    ]
    .fillna(
        "SIN CLASIFICAR"
    )
)


# ==========================================================
# IMPORTE ANALÍTICO
# ==========================================================

# En la contabilidad original: los ingresos suelen venir negativos y los gastos en positivo.
# Se invierte el signo para la PyG, conservando anulaciones y reclasificaciones.

contabilidad_financiera[
    "importe_analitico"
] = np.where(
    contabilidad_financiera[
        "tipo_estado"
    ].eq("PyG"),
    -contabilidad_financiera[
        "importe"
    ],
    contabilidad_financiera[
        "importe"
    ]
)


# ==========================================================
# VALIDACIÓN
# ==========================================================

if len(contabilidad_financiera) != len(contabilidad):

    raise ValueError(
        "La integración con el maestro ha alterado "
        "el número de movimientos contables."
    )

movimientos_sin_clasificar = (
    contabilidad_financiera[
        "estado_clasificacion"
    ]
    .eq("SIN CLASIFICAR")
    .sum()
)

print(
    f"Filas integradas: "
    f"{len(contabilidad_financiera):,}"
)

print(
    f"Movimientos sin clasificar: "
    f"{movimientos_sin_clasificar:,}"
)

display(
    contabilidad_financiera.head()
)

Filas integradas: 78,688
Movimientos sin clasificar: 0


,periodo,año,mes_numero,año_mes,trimestre,cuenta_n0,cuenta_n1,cuenta_n2,cuenta_n5,centro_coste,intercompany,importe,importe_debe,importe_haber,descripcion_n5,tipo_estado,masa_financiera,categoria_financiera,subcategoria_financiera,naturaleza,circulante,driver_negocio,estado_clasificacion,importe_analitico
0,2021-01-01,2021,1,2021-01,T1,2,21,213,213000,S002,NO,0.00,0.00,0.00,Maquinaria (slots machines),Balance,Activo no corriente,Inmovilizado,Inmovilizado material,Deudora,No corriente,Inversión,CLASIFICADA,0.00
1,2021-01-01,2021,1,2021-01,T1,2,21,213,213000,S002,i1,"-96,800.00",0.00,"96,800.00",Maquinaria (slots machines),Balance,Activo no corriente,Inmovilizado,Inmovilizado material,Deudora,No corriente,Inversión,CLASIFICADA,"-96,800.00"
2,2021-01-01,2021,1,2021-01,T1,2,21,213,213000,S002,i2,"-7,621.38",0.00,"7,621.00",Maquinaria (slots machines),Balance,Activo no corriente,Inmovilizado,Inmovilizado material,Deudora,No corriente,Inversión,CLASIFICADA,"-7,621.38"
3,2021-01-01,2021,1,2021-01,T1,2,21,213,213000,S003,NO,"6,171.00","6,171.00",0.00,Maquinaria (slots machines),Balance,Activo no corriente,Inmovilizado,Inmovilizado material,Deudora,No corriente,Inversión,CLASIFICADA,"6,171.00"
4,2021-02-01,2021,2,2021-02,T1,2,21,213,213000,S003,NO,622.04,622.00,0.00,Maquinaria (slots machines),Balance,Activo no corriente,Inmovilizado,Inmovilizado material,Deudora,No corriente,Inversión,CLASIFICADA,622.04


In [6]:
# ==========================================================
# 6. CUENTAS A REVISAR
# ==========================================================

cuentas_revision = (
    contabilidad_financiera.loc[
        contabilidad_financiera[
            "estado_clasificacion"
        ].ne("CLASIFICADA")
    ]
    .groupby(
        [
            "cuenta_n5",
            "descripcion_n5",
            "tipo_estado",
            "masa_financiera",
            "categoria_financiera",
            "estado_clasificacion",
        ],
        as_index=False,
        dropna=False,
        observed=True
    )
    .agg(
        numero_movimientos=(
            "importe",
            "size"
        ),
        importe_total=(
            "importe",
            "sum"
        ),
        primera_fecha=(
            "periodo",
            "min"
        ),
        ultima_fecha=(
            "periodo",
            "max"
        ),
    )
)

if not cuentas_revision.empty:

    cuentas_revision = (
        cuentas_revision
        .sort_values(
            "importe_total",
            key=lambda x: x.abs(),
            ascending=False
        )
        .reset_index(drop=True)
    )

print(
    f"Cuentas utilizadas a revisar: "
    f"{len(cuentas_revision):,}"
)

display(
    cuentas_revision.head(20)
)

Cuentas utilizadas a revisar: 0


,cuenta_n5,descripcion_n5,tipo_estado,masa_financiera,categoria_financiera,estado_clasificacion,numero_movimientos,importe_total,primera_fecha,ultima_fecha


In [7]:
# ==========================================================
# 7. RESUMEN Y CONTROL DE CALIDAD
# ==========================================================

# ==========================================================
# RESUMEN DE CLASIFICACIÓN
# ==========================================================

resumen_clasificacion = (
    contabilidad_financiera
    .groupby(
        [
            "tipo_estado",
            "masa_financiera",
            "categoria_financiera",
            "estado_clasificacion",
        ],
        as_index=False,
        observed=True,
        dropna=False
    )
    .agg(
        numero_movimientos=(
            "importe",
            "size"
        ),
        importe_total=(
            "importe",
            "sum"
        ),
    )
    .sort_values(
        [
            "tipo_estado",
            "masa_financiera",
            "categoria_financiera",
        ]
    )
    .reset_index(drop=True)
)


# ==========================================================
# CONTROL GENERAL
# ==========================================================

control_nb2 = pd.DataFrame([
    [
        "filas_integradas",
        len(contabilidad_financiera)
    ],
    [
        "cuentas_utilizadas",
        contabilidad_financiera[
            "cuenta_n5"
        ].nunique()
    ],
    [
        "cuentas_maestro",
        len(maestro_financiero)
    ],
    [
        "cuentas_utilizadas_a_revisar",
        cuentas_revision[
            "cuenta_n5"
        ].nunique()
    ],
    [
        "movimientos_sin_clasificar",
        movimientos_sin_clasificar
    ],
    [
        "importe_en_cuentas_a_revisar",
        (
            cuentas_revision[
                "importe_total"
            ]
            .abs()
            .sum()
            if not cuentas_revision.empty
            else 0.0
        )
    ],
], columns=[
    "indicador",
    "valor"
])


display(
    control_nb2
)

display(
    resumen_clasificacion.head(20)
)

,indicador,valor
0,filas_integradas,"78,688.00"
1,cuentas_utilizadas,240.00
2,cuentas_maestro,"1,565.00"
3,cuentas_utilizadas_a_revisar,0.00
4,movimientos_sin_clasificar,0.00
5,importe_en_cuentas_a_revisar,0.00


,tipo_estado,masa_financiera,categoria_financiera,estado_clasificacion,numero_movimientos,importe_total
0,Balance,Activo corriente,Administraciones públicas deudoras,CLASIFICADA,4044,"-44,342.59"
1,Balance,Activo corriente,Clientes y deudores,CLASIFICADA,392,"25,240.82"
2,Balance,Activo corriente,Cuentas con vinculadas,CLASIFICADA,278,"217,664.42"
3,Balance,Activo corriente,Fianzas y depósitos,CLASIFICADA,3,450.00
4,Balance,Activo corriente,Periodificaciones,CLASIFICADA,966,927.75
5,Balance,Activo corriente,Personal,CLASIFICADA,210,"23,082.84"
6,Balance,Activo corriente,Tesorería,CLASIFICADA,2328,"1,623,637.54"
7,Balance,Activo no corriente,Correcciones de activo,CLASIFICADA,7830,"-4,565,830.38"
8,Balance,Activo no corriente,Inmovilizado,CLASIFICADA,611,"2,902,480.24"
9,Balance,Activo no corriente,Inversiones financieras,CLASIFICADA,27,"-1,158,762.02"


In [8]:
# ==========================================================
# 8. EXPORTACIÓN
# ==========================================================

EXPORTACIONES = {
    "02_maestro_financiero.csv":
        maestro_financiero,

    "02_contabilidad_financiera.csv":
        contabilidad_financiera,

    "02_cuentas_revision.csv":
        cuentas_revision,

    "02_resumen_clasificacion.csv":
        resumen_clasificacion,

    "02_control_calidad.csv":
        control_nb2,
}

for nombre, df in EXPORTACIONES.items():

    df.to_csv(
        OUTPUT_DIR / nombre,
        index=False,
        encoding="utf-8-sig",
        decimal="."
    )

print("Archivos exportados:")

for nombre in EXPORTACIONES:

    print(
        f" - data/processed/02/{nombre}"
    )

print(
    f"\nCuentas en maestro: "
    f"{len(maestro_financiero):,}"
)

print(
    f"Movimientos integrados: "
    f"{len(contabilidad_financiera):,}"
)

print(
    f"Cuentas utilizadas a revisar: "
    f"{cuentas_revision['cuenta_n5'].nunique():,}"
)

print(
    f"Movimientos sin clasificar: "
    f"{movimientos_sin_clasificar:,}"
)

Archivos exportados:
 - data/processed/02/02_maestro_financiero.csv
 - data/processed/02/02_contabilidad_financiera.csv
 - data/processed/02/02_cuentas_revision.csv
 - data/processed/02/02_resumen_clasificacion.csv
 - data/processed/02/02_control_calidad.csv

Cuentas en maestro: 1,565
Movimientos integrados: 78,688
Cuentas utilizadas a revisar: 0
Movimientos sin clasificar: 0


## Salidas principales

- `02_maestro_financiero.csv`: una fila por cuenta contable y su clasificación financiera.
- `02_contabilidad_financiera.csv`: movimientos contables enriquecidos con la clasificación.
- `02_cuentas_revision.csv`: cuentas ambiguas o no clasificadas que deben validarse.
- `02_resumen_clasificacion.csv`: resumen de movimientos e importes por categoría.
- `02_control_calidad.csv`: control general del proceso.